# Information Entropy of $m$-Blocks — Worked Examples
## Constructing a set of $m$-blocks, counting it, and measuring it

> 📺 **[Watch the original lecture on YouTube](https://youtu.be/GNV5RXf_uSA)**

[Part 3](../Video_03/Information_Entropy_of_m_blocks.ipynb) defined the $m$-block and derived
$\Omega = U^{\,m}$. This lecture puts that machinery to work on a concrete message. Three questions
get answered:

1. **How do you actually build a set of $m$-blocks** from a set of single characters?
2. **How many blocks do you end up with?** This has a formula:
   $$N_m = 1 + \frac{N - m}{SS}$$
3. **What is the information entropy of the resulting set**, and how does it compare with the
   entropy of the original single-character message?

The message used throughout is the same 10-bit string from
[Part 2](../Video_02/Information_Entropy_Examples.ipynb), which lets us put the block result and
the single-character result side by side at the end.

That final comparison is the interesting part of this lecture — and the part that rewards careful
attention to units. Section 9 works through it.

> 🧭 **Navigation** · [← Part 3: Information Entropy of m-Blocks](../Video_03/Information_Entropy_of_m_blocks.ipynb) · **Part 4 (current)** · [Part 5: Information Entropy of a Genome →](../Video_05/Information_Entropy_of_a_Genome.ipynb)

---

## 1. Recap — where Part 3 left off

Part 3 established the counting relationship at the centre of all of this. Given a message of $N$
single characters drawn from an alphabet of $U$ distinct characters, grouping them into blocks of
$m$ gives a **new alphabet** whose size is

$$\Omega = U^{\,m}$$

and the Shannon formula applies to that new alphabet unchanged — only the upper limit of the sum
moves from $U$ to $\Omega$:

$$H(\mathcal{X})^{(m)} = -\sum_{j=1}^{\Omega} p_j^{(m)} \log_2 p_j^{(m)}
\quad \text{[bits per } m\text{-block]}$$

What Part 3 did *not* do is build such a set from an actual message. That is the gap this lecture
closes.

| Symbol | Meaning |
|---|---|
| $N$ | number of single characters in the original message |
| $U$ | number of distinct single characters |
| $m$ | block size, with $1 \leq m \leq N$ |
| $SS$ | step size (how far the window slides), with $1 \leq SS \leq m$ |
| $\Omega$ | number of **distinct** $m$-blocks possible, $= U^{\,m}$ |
| $N_m$ | number of $m$-blocks actually **generated** from the message |

Note the distinction between the last two, because the rest of the notebook turns on it.
$\Omega$ counts what is *possible*; $N_m$ counts what you actually *produced*. They are different
numbers and they answer different questions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def shannon_entropy(probabilities, base=2):
    """Shannon entropy H = -sum(p * log_b(p)).  Skips zero-probability terms."""
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    assert np.isclose(p.sum(), 1.0), f"Probabilities must sum to 1 (got {p.sum()})"
    return -np.sum(p * np.log(p) / np.log(base))

def entropy_of_message(message, base=2):
    """Compute H(X) directly from a sequence of symbols (string or list)."""
    N = len(message)
    counts = Counter(message)
    probs = [c / N for c in counts.values()]
    return shannon_entropy(probs, base=base), counts

# The message from Part 2, reused throughout this lecture
message = [0, 1, 1, 1, 1, 0, 1, 1, 0, 1]
N = len(message)
U = len(set(message))

print(f"message = {message}")
print(f"N = {N} single characters")
print(f"U = {U} distinct characters {sorted(set(message))}")

---
## 2. Constructing a set of $m$-blocks

The procedure is mechanical. Put a window of width $m$ at the start of the message, read off the
$m$ characters inside it as one block, then slide the window to the right by the **step size** $SS$
and repeat until you run out of message.

Two parameters control it, and each has a range:

$$1 \leq m \leq N \qquad\qquad 1 \leq SS \leq m$$

The upper bound on $SS$ is the one that matters. If $SS > m$ the window jumps further than its own
width, so characters fall into the gap between one block and the next and are **never read at
all**. Keeping $SS \leq m$ guarantees every character of the original message ends up in at least
one block.

### Worked construction — $m = 2$, $SS = 2$

Take the message and set both the block size and the step size to 2:

$$\underbrace{0\,1}_{01}\;\underbrace{1\,1}_{11}\;\underbrace{1\,0}_{10}\;\underbrace{1\,1}_{11}\;\underbrace{0\,1}_{01}$$

giving the new set

$$\{01,\; 11,\; 10,\; 11,\; 01\}$$

Ten single characters have become five two-character blocks. Because $SS = m$ here, the blocks
**tile** the message: they sit end to end, no gaps and no overlap, and every original character is
used exactly once.

### Presenter's Notes — Constructing the set of $m$-blocks by sliding a window
![Board showing the message split into five two-character blocks with step size 2](Screenshots/Part_4_00_06_45.jpg)

*Screenshot at timestamp 00:06:45*

---

In [ ]:
def make_blocks(sequence, m, SS=None):
    """Slide a width-m window along `sequence`, advancing by SS each step.

    SS defaults to m (the tiling case). The lecture's condition is 1 <= SS <= m;
    a larger step would skip characters between consecutive blocks.
    """
    SS = m if SS is None else SS
    if not 1 <= m <= len(sequence):
        raise ValueError(f"block size must satisfy 1 <= m <= N (got m={m}, N={len(sequence)})")
    if not 1 <= SS <= m:
        raise ValueError(f"step size must satisfy 1 <= SS <= m (got SS={SS}, m={m})")
    seq = list(sequence)
    return [tuple(seq[i:i + m]) for i in range(0, len(seq) - m + 1, SS)]

def as_text(blocks):
    """Render a list of block tuples as compact strings for printing."""
    return [''.join(map(str, b)) for b in blocks]

blocks_ss2 = make_blocks(message, m=2, SS=2)
print(f"message           : {''.join(map(str, message))}   (N = {N})")
print(f"m = 2, SS = 2     : {as_text(blocks_ss2)}")
print(f"N_m               : {len(blocks_ss2)} blocks")
print()
print(f"characters consumed: {len(blocks_ss2) * 2} block-slots covering {N} characters "
      f"-> each character used exactly once")

---
## 3. How many blocks? The formula for $N_m$

Counting five blocks by eye is fine for a 10-character message. For a real one we want a formula
relating $N_m$ to $N$, $m$ and $SS$. The lecture gives:

$$\boxed{\;N_m = 1 + \frac{N - m}{SS}\;}$$

The reasoning behind it: the first block is free — place the window at the start and you have one
block, which is where the leading $1$ comes from. That consumes $m$ characters, leaving $N - m$
still to the right of the window. Every further slide of $SS$ characters yields one more block, so
the remaining stretch contributes $(N-m)/SS$ additional blocks.

### Checking it against the construction above

With $N = 10$, $m = 2$, $SS = 2$:

$$N_m = 1 + \frac{10 - 2}{2} = 1 + \frac{8}{2} = 1 + 4 = 5$$

which is exactly the five blocks we counted.

> **On the provenance of this formula.** Dr. Vopson notes in the lecture that the version printed
> in his book *Reality Reloaded* — and in his original patent submission — was different, and
> works only in limited circumstances. The form above came out of a collaboration with a colleague
> he names as Simon, who spotted the correction while they were improving the patent. He is
> explicit that this is the version that holds for any combination of $N$, $m$ and $SS$.

### Presenter's Notes — The formula $N_m = 1 + (N-m)/SS$, and testing it
![Board showing the boxed formula for N_m and its numerical check giving 5](Screenshots/Part_4_00_09_50.jpg)

*Screenshot at timestamp 00:09:50*

---

In [ ]:
def N_m_formula(N, m, SS):
    """The lecture's formula: N_m = 1 + (N - m) / SS."""
    return 1 + (N - m) / SS

print(f"{'N':>4} {'m':>3} {'SS':>3}   {'formula':>9}   {'counted':>8}   match")
print("-" * 48)
for m, SS in [(2, 2), (2, 1), (3, 3), (3, 1), (4, 4), (4, 2), (5, 5), (1, 1)]:
    predicted = N_m_formula(N, m, SS)
    actual = len(make_blocks(message, m, SS))
    print(f"{N:>4} {m:>3} {SS:>3}   {predicted:>9.2f}   {actual:>8}   "
          f"{'yes' if np.isclose(predicted, actual) else 'NO'}")

### Where the formula needs a floor

Most rows match — but two do not, and they are worth looking at: $m=3,\,SS=3$ predicts $3.33$
against 3 actual blocks, and $m=4,\,SS=4$ predicts $2.5$ against 2.

The pattern is that in both, $SS$ does not divide $N-m$ evenly, so the formula returns a fraction.
There is no such thing as a fractional block: the window runs out of message before it can complete
another full step, and that final partial step produces nothing. The general form therefore takes
the floor:

$$N_m = \left\lfloor \frac{N - m}{SS} \right\rfloor + 1$$

This reduces to the lecture's formula exactly when $SS \mid (N-m)$ — which is the case for both
worked examples in this lecture ($SS = 2$ and $SS = 1$ with $N - m = 8$), so nothing in the
lecture's arithmetic is affected. The floor only matters when you apply the formula to arbitrary
values of $m$ and $SS$, as the cell below shows.

In [ ]:
def N_m_floor(N, m, SS):
    """General form: the window must complete a full step to produce another block."""
    return (N - m) // SS + 1

print("Combinations where SS does not divide (N - m):\n")
print(f"{'m':>3} {'SS':>3}   {'1+(N-m)/SS':>11}   {'floor form':>10}   {'counted':>8}")
print("-" * 44)
any_frac = False
for m in range(1, N + 1):
    for SS in range(1, m + 1):
        raw = N_m_formula(N, m, SS)
        if not float(raw).is_integer():
            any_frac = True
            print(f"{m:>3} {SS:>3}   {raw:>11.2f}   {N_m_floor(N, m, SS):>10}   "
                  f"{len(make_blocks(message, m, SS)):>8}")
if not any_frac:
    print("(none)")

# the floor form is correct for every valid (m, SS) on this message
assert all(
    N_m_floor(N, m, SS) == len(make_blocks(message, m, SS))
    for m in range(1, N + 1) for SS in range(1, m + 1)
)
print("\nFloor form verified against direct construction for all valid (m, SS).")

---
## 4. Changing the step size — $m = 2$, $SS = 1$

The step size is a free choice within $1 \leq SS \leq m$. With $m = 2$ there are exactly two
options: $SS = 2$, which we just did, and $SS = 1$. Taking the smaller step means the window
advances one character at a time, so consecutive blocks **overlap**.

The formula predicts

$$N_m = 1 + \frac{10 - 2}{1} = 1 + 8 = 9$$

and sliding the window one character at a time through $0\,1\,1\,1\,1\,0\,1\,1\,0\,1$ gives

$$\{01,\; 11,\; 11,\; 11,\; 10,\; 01,\; 11,\; 10,\; 01\}$$

Nine blocks — nearly as many elements as the ten single characters we started with. The lecture
draws attention to this: a smaller step extracts far more blocks from the same message. Each
character now appears in up to two different blocks, which is worth holding onto for Section 9.

### Presenter's Notes — $SS = 1$ gives $N_m = 9$ overlapping blocks
![Board showing the nine overlapping two-character blocks for step size 1](Screenshots/Part_4_00_14_35.jpg)

*Screenshot at timestamp 00:14:35*

---

In [ ]:
blocks_ss1 = make_blocks(message, m=2, SS=1)

print(f"message        : {''.join(map(str, message))}")
print(f"m = 2, SS = 1  : {as_text(blocks_ss1)}")
print(f"N_m            : {len(blocks_ss1)}   (formula: 1 + (10-2)/1 = {N_m_formula(N, 2, 1):.0f})")
print()
print(f"{'':16}{'SS = 2':>12}{'SS = 1':>12}")
print("-" * 40)
print(f"{'blocks (N_m)':16}{len(blocks_ss2):>12}{len(blocks_ss1):>12}")
print(f"{'block-slots':16}{len(blocks_ss2) * 2:>12}{len(blocks_ss1) * 2:>12}")
print(f"{'covering N':16}{N:>12}{N:>12}")
print(f"{'each char used':16}{len(blocks_ss2) * 2 / N:>11.1f}x{len(blocks_ss1) * 2 / N:>11.1f}x")

---
## 5. The block alphabet and its probability distribution

With $U = 2$ and $m = 2$, Part 3's counting relationship gives the size of the new alphabet:

$$\Omega = U^{\,m} = 2^2 = 4$$

so the set of distinct $m$-blocks is

$$\mathcal{X} = \{00,\; 01,\; 10,\; 11\}$$

and each carries a probability, forming the distribution

$$P = \{p_{00},\; p_{01},\; p_{10},\; p_{11}\}$$

Counting occurrences in the nine-block set $\{01, 11, 11, 11, 10, 01, 11, 10, 01\}$:

| Block | Occurrences | Probability |
|---|---|---|
| $00$ | 0 | $p_{00} = 0$ |
| $01$ | 3 | $p_{01} = 3/9$ |
| $10$ | 2 | $p_{10} = 2/9$ |
| $11$ | 4 | $p_{11} = 4/9$ |

Note that $00$ **never occurs** — the message contains no two adjacent zeros. This is the concrete
version of a point from Part 3: $\Omega$ counts the blocks that are *possible*, not the ones that
appear. A block with $p = 0$ contributes nothing to the entropy sum, so in effect only three of the
four available states are doing any work here.

In [ ]:
counts_ss1 = Counter(blocks_ss1)
Omega = U ** 2
alphabet = [(0, 0), (0, 1), (1, 0), (1, 1)]

print(f"Omega = U^m = {U}^2 = {Omega} possible distinct 2-blocks")
print(f"X = {{{', '.join(as_text(alphabet))}}}\n")
print(f"{'block':>6}{'count':>8}{'probability':>14}")
print("-" * 28)
for b in alphabet:
    c = counts_ss1.get(b, 0)
    print(f"{''.join(map(str, b)):>6}{c:>8}{f'{c}/9 = {c / 9:.4f}':>14}")

observed = sum(1 for b in alphabet if counts_ss1.get(b, 0) > 0)
print(f"\ntotal: {sum(counts_ss1.values())} blocks")
print(f"{observed} of the {Omega} possible blocks actually occur "
      f"-- '00' never appears (no two adjacent zeros in the message)")

---
## 6. The information entropy of the set of $m$-blocks

The formula is Part 3's, unchanged. Sum over the $\Omega = 4$ possible blocks:

$$H(\mathcal{X})^{(m)} = -\sum_{j=1}^{\Omega} p_j^{(m)} \log_2 p_j^{(m)}$$

The $p_{00} = 0$ term drops out — $0\log_2 0 = 0$ — leaving three terms:

$$H(\mathcal{X})^{(m)} = -\left[\tfrac{3}{9}\log_2\tfrac{3}{9}
+ \tfrac{2}{9}\log_2\tfrac{2}{9}
+ \tfrac{4}{9}\log_2\tfrac{4}{9}\right]$$

$$H(\mathcal{X})^{(m)} = 1.53 \text{ bits per } m\text{-block}$$

against a ceiling, from Part 3's scaling law, of

$$H(\mathcal{X})^{(m)}_{\max} = \log_2 \Omega = m \cdot H(\mathcal{X})_{\max} = 2 \times 1 = 2
\text{ bits per } m\text{-block}$$

So each two-character block carries 1.53 of a possible 2 bits — about 77% of the maximum. The
shortfall is the non-uniformity of the distribution, and in particular the state that never occurs.

### Presenter's Notes — The probability distribution and $H(\mathcal{X})^{(m)} = 1.53$ bits
![Board showing the probability distribution and the entropy calculation giving 1.53 bits](Screenshots/Part_4_00_20_30.jpg)

*Screenshot at timestamp 00:20:30*

---

In [ ]:
H_block_ss1, _ = entropy_of_message(blocks_ss1)
H_block_max = np.log2(Omega)

print(f"H(X)^(m)      = {H_block_ss1:.4f} bits per m-block   (lecture: 1.53)")
print(f"H(X)^(m)_max  = log2(Omega) = log2({Omega}) = {H_block_max:.4f} bits per m-block")
print(f"efficiency    = {H_block_ss1 / H_block_max:.1%} of the maximum")
print()

# reproduce the three-term sum exactly as written on the board
terms = [3 / 9, 2 / 9, 4 / 9]
by_hand = -sum(p * np.log2(p) for p in terms)
print(f"term by term  : -[3/9*log2(3/9) + 2/9*log2(2/9) + 4/9*log2(4/9)] = {by_hand:.4f}")
assert np.isclose(by_hand, H_block_ss1), "hand calculation disagrees with the counter"
print("matches the direct computation.")

---
## 7. Total information content of the set

Entropy is a *per-character* quantity. To get the information content of the whole set, multiply by
the number of characters in it — here, the number of $m$-blocks:

$$\text{Inf} = N_m \cdot H(\mathcal{X})^{(m)} = 9 \times 1.53 = 13.77 \text{ bits}$$

$$\text{Inf}_{\max} = N_m \cdot H(\mathcal{X})^{(m)}_{\max} = 9 \times 2 = 18 \text{ bits}$$

So the nine-block set encodes 13.77 bits where 18 were available.

### The comparison the lecture draws

Now put that beside the single-character result computed in
[Part 2](../Video_02/Information_Entropy_Examples.ipynb) for the *same* message:

| | single characters | $m$-blocks ($m=2$, $SS=1$) |
|---|---|---|
| set size | $N = 10$ | $N_m = 9$ |
| entropy | $H(\mathcal{X}) = 0.881$ bits | $H(\mathcal{X})^{(m)} = 1.53$ bits |
| maximum | $H_{\max} = 1$ bit | $H^{(m)}_{\max} = 2$ bits |
| total information | $10 \times 0.881 = 8.81$ bits | $9 \times 1.53 = 13.77$ bits |
| maximum possible | 10 bits | 18 bits |

Both entropy per element and total information came out larger for the block set. The lecture reads
this as an argument that if your goal is to *minimise* the information content of a message — to
compress it — then the smallest denomination, single characters, is the better choice.

### Presenter's Notes — The full comparison — block set against single-character set
![Board comparing 13.77 bits for the block set against 8.81 bits for single characters](Screenshots/Part_4_00_24_25.jpg)

*Screenshot at timestamp 00:24:25*

---

In [ ]:
H_single, _ = entropy_of_message(message)
H_single_max = np.log2(U)

rows = [
    ("set size",            f"N = {N}",                    f"N_m = {len(blocks_ss1)}"),
    ("entropy per element", f"{H_single:.4f} bits",        f"{H_block_ss1:.4f} bits"),
    ("maximum per element", f"{H_single_max:.4f} bits",    f"{H_block_max:.4f} bits"),
    ("total information",   f"{N * H_single:.2f} bits",    f"{len(blocks_ss1) * H_block_ss1:.2f} bits"),
    ("maximum possible",    f"{N * H_single_max:.2f} bits", f"{len(blocks_ss1) * H_block_max:.2f} bits"),
]

print(f"{'':22}{'single characters':>20}{'m-blocks (SS=1)':>20}")
print("-" * 62)
for label, a, b in rows:
    print(f"{label:22}{a:>20}{b:>20}")

---
## 8. Reading that comparison carefully

Every number in the table above is correct as computed. But two of the rows compare quantities in
**different units**, and it is worth being precise about what they do and do not show. This is the
single most instructive point in the lecture, so it is worth slowing down on.

### The units differ: per block is not per character

$H(\mathcal{X}) = 0.881$ is measured in **bits per single character**. $H(\mathcal{X})^{(m)} = 1.53$
is measured in **bits per two-character block**. Each block contains two characters, so to compare
them on a common footing we use the per-character block entropy from Part 3:

$$h_m = \frac{H(\mathcal{X})^{(m)}}{m} = \frac{1.53}{2} = 0.765 \text{ bits per character}$$

which is *smaller* than $0.881$, not larger. This is the direction Part 3's theory predicts:
$h_m$ is non-increasing in $m$, because a larger block can capture correlations between neighbouring
characters that single-character counting is blind to.

### The totals differ because overlapping blocks reuse characters

With $SS = 1$ the nine blocks span $9 \times 2 = 18$ character-slots, but the message has only 10
characters — so each interior character is read into **two** different blocks. The 13.77 bits is
counting most of the message roughly twice. That is not a flaw in the arithmetic; it is what
$SS < m$ means.

The clean comparison uses $SS = m = 2$, where the blocks tile the message and every character is
used exactly once — the same 10 characters, counted once, in both columns.

In [ ]:
H_block_ss2, _ = entropy_of_message(blocks_ss2)

h_per_char_ss1 = H_block_ss1 / 2
h_per_char_ss2 = H_block_ss2 / 2

print("PER-CHARACTER ENTROPY (the like-for-like comparison)")
print("-" * 58)
print(f"  m = 1 (single characters)      h_1 = {H_single:.4f} bits/character")
print(f"  m = 2, SS = 1 (overlapping)    h_2 = {H_block_ss1:.4f}/2 = {h_per_char_ss1:.4f} bits/character")
print(f"  m = 2, SS = 2 (tiling)         h_2 = {H_block_ss2:.4f}/2 = {h_per_char_ss2:.4f} bits/character")
print()
print(f"  blocking lowers per-character entropy by "
      f"{H_single - h_per_char_ss2:.4f} bits ({(H_single - h_per_char_ss2) / H_single:.1%})")
print()

total_single = N * H_single
total_ss2 = len(blocks_ss2) * H_block_ss2
total_ss1 = len(blocks_ss1) * H_block_ss1

print("TOTAL INFORMATION, each covering the same 10-character message")
print("-" * 58)
print(f"  single characters   {N} x {H_single:.4f} = {total_single:>6.2f} bits   "
      f"({N} character-slots)")
print(f"  m = 2, SS = 2       {len(blocks_ss2)} x {H_block_ss2:.4f} = {total_ss2:>6.2f} bits   "
      f"({len(blocks_ss2) * 2} character-slots -- each character once)")
print(f"  m = 2, SS = 1       {len(blocks_ss1)} x {H_block_ss1:.4f} = {total_ss1:>6.2f} bits   "
      f"({len(blocks_ss1) * 2} character-slots -- each character ~1.8x)")
print()
print(f"  Against a true tiling, blocking REDUCES the total from "
      f"{total_single:.2f} to {total_ss2:.2f} bits.")
print(f"  The {total_ss1:.2f} figure is larger because SS = 1 reads most characters twice.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# --- left: per-character entropy ---
labels = ['$m=1$\nsingle chars', '$m=2$, $SS=1$\noverlapping', '$m=2$, $SS=2$\ntiling']
vals = [H_single, h_per_char_ss1, h_per_char_ss2]
axes[0].bar(labels, vals, color=['steelblue', 'darkorange', 'seagreen'], edgecolor='white')
axes[0].axhline(H_single_max, color='crimson', linestyle='--', linewidth=1.5,
                label=f'$H_{{max}}$ = {H_single_max:.0f} bit/character')
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('bits per character', fontsize=11)
axes[0].set_title('Per-character entropy $h_m = H^{(m)}/m$\nblocking lowers it, as Part 3 predicts',
                  fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# --- right: total information ---
tot_labels = ['$m=1$\n10 slots', '$m=2$, $SS=2$\n10 slots', '$m=2$, $SS=1$\n18 slots']
tot_vals = [total_single, total_ss2, total_ss1]
bars = axes[1].bar(tot_labels, tot_vals, color=['steelblue', 'seagreen', 'darkorange'],
                   edgecolor='white')
bars[2].set_hatch('//')
for i, v in enumerate(tot_vals):
    axes[1].text(i, v + 0.25, f'{v:.2f}', ha='center', fontsize=10)
axes[1].set_ylabel('total bits', fontsize=11)
axes[1].set_title('Total information for the same 10-character message\n'
                  'the hatched bar counts most characters twice', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### What this does and does not settle

Two conclusions survive the unit check, and one needs qualifying.

- **The lecture's headline is right about the block *set*.** A set of nine overlapping 2-blocks
  genuinely does carry more total information than the original ten characters, because it is a
  larger, partly-redundant representation of the same message. If you build that set, you have made
  something bigger.
- **But blocking itself is not what inflates it — overlap is.** With $SS = m$ the same operation
  *reduces* the total, from 8.81 bits to 7.61.
- **So "smallest denomination is best for compression" needs care.** On a like-for-like basis
  ($SS = m$, per character) larger blocks compress *better*, not worse, exactly as Part 3's
  entropy-rate argument requires — and real compressors exploit precisely this.

> **Confirmed by the lecturer.** Part 5 of this series opens with exactly this clarification: the increase in total information holds *"only if the step size of generating the m blocks is smaller than the m block itself … because you are overlapping the m blocks and counting many characters multiple times."* This section was written before that lecture was available; see [Part 5, Section 1](../Video_05/Information_Entropy_of_a_Genome.ipynb) for his own statement of it.

One honest caveat on all of these numbers. A 10-character message yields 5 or 9 blocks against
$\Omega = 4$ possible states, so these are extremely small samples and the entropy estimates carry
substantial finite-sample bias — the effect quantified in
[Part 3, Section 11](../Video_03/Information_Entropy_of_m_blocks.ipynb). They are exactly right as
arithmetic on this message, and should not be read as precise estimates of any underlying source.

---
## 9. So why does computing use bytes?

If single characters minimise information content, why is essentially all digital encoding built on
the byte — an $m$-block of 8 bits?

Because compression is not the only objective. The other is having enough distinct states to
represent everything you need to write down. Part 3's counting relationship answers that directly:

$$\Omega = U^{\,m} = 2^8 = 256$$

Two hundred and fifty-six distinct states is enough to give a unique code to every capital letter,
every lower-case letter, the digits, punctuation, mathematical symbols and assorted control
characters — with room to spare. An $m$-block of 1 offers two states, which encodes nothing on its
own.

So block size is a **trade-off**, and the lecture states both sides of it plainly:

| Larger $m$ | Smaller $m$ |
|---|---|
| more distinct states $\Omega = U^m$ | fewer states |
| richer representation — letters, symbols, control codes | little expressive capacity alone |
| more total information to store or transmit | less total information |

You buy expressive range with storage.

### Presenter's Notes — The byte as an $m$-block of 8: $\Omega = 2^8 = 256$
![Board showing the powers of two building toward 256 states for a byte](Screenshots/Part_4_00_27_10.jpg)

*Screenshot at timestamp 00:27:10*

---

In [ ]:
print(f"{'m':>3}{'Omega = 2^m':>14}   what it can represent")
print("-" * 62)
notes = {
    1: "a single bit -- two states",
    2: "four states -- the size of the DNA alphabet",
    3: "eight states",
    4: "one hexadecimal digit",
    7: "the original ASCII range",
    8: "one byte -- letters, digits, punctuation, control codes",
    16: "65,536 states -- covers most of Unicode's basic plane",
}
for m in (1, 2, 3, 4, 7, 8, 16):
    print(f"{m:>3}{2 ** m:>14,}   {notes[m]}")

print()
print("Maximum entropy scales only linearly with m (Part 3):")
print(f"{'m':>3}{'Omega':>12}{'H_max = log2(Omega)':>22}")
print("-" * 38)
for m in (1, 2, 4, 8, 16):
    print(f"{m:>3}{2 ** m:>12,}{np.log2(2 ** m):>18.0f} bits")

---
## 10. A closing aside — why four letters?

The lecture ends on an open question rather than a result, and it is worth recording as such.

Genomic information is encoded in an alphabet of four nucleotides, $\{A, C, T, G\}$. Over a binary
alphabet that is an $m$-block of 2:

$$\Omega = U^{\,m} = 2^2 = 4$$

If minimal information content were the only objective, $m = 1$ — a binary code — would have been
the choice. Life did not take it. Dr. Vopson's suggestion is that four letters buy **diversity of
expression**: more distinct states per position, and so more room to encode the variety seen in
living systems, at the cost of carrying more information.

He is careful to present this as an interesting question rather than a settled answer — explicitly
declining to claim intelligent design, and allowing that it may simply be how things fell into
place. It is flagged here in the same spirit.

The next lecture takes this up concretely, applying $m$-blocks to genomic sequences.

---
## Summary

| Quantity | Formula | This lecture's example |
|---|---|---|
| Step size | $1 \leq SS \leq m$ | $SS = 2$, then $SS = 1$ |
| Blocks generated | $N_m = 1 + \dfrac{N-m}{SS}$ | $5$, then $9$ |
| Distinct blocks possible | $\Omega = U^{\,m}$ | $2^2 = 4$ |
| Block entropy | $H(\mathcal{X})^{(m)} = -\sum_{j=1}^{\Omega} p_j^{(m)}\log_2 p_j^{(m)}$ | $1.53$ bits per block |
| Maximum | $H^{(m)}_{\max} = \log_2\Omega = m\,H_{\max}$ | $2$ bits per block |
| Total information | $\text{Inf} = N_m \cdot H(\mathcal{X})^{(m)}$ | $13.77$ bits |

### Key takeaways

1. **Building a block set is mechanical**: slide a width-$m$ window in steps of $SS$. The condition
   $SS \leq m$ is what guarantees no character is skipped.
2. **$N_m = 1 + (N-m)/SS$** counts what you generate — one block for free, then one per step. Take
   the floor when $SS$ does not divide $N-m$.
3. **$N_m$ and $\Omega$ are different things.** $\Omega = U^m$ is how many blocks are *possible*;
   $N_m$ is how many you *produced*. Here $\Omega = 4$ while $N_m = 9$, and one of the four possible
   blocks never occurred at all.
4. **The entropy formula does not change** — only the upper limit of the sum, from $U$ to $\Omega$.
5. **Compare per character, not per block.** $1.53$ bits per 2-block is $0.765$ bits per character,
   *below* the $0.881$ of single characters — the direction Part 3 predicts.
6. **Overlap inflates totals.** $SS = 1$ reads most characters twice, which is why the block set
   totals more bits. With $SS = m$ the total falls from $8.81$ to $7.61$ bits.
7. **Block size is a trade-off**: $\Omega = U^m$ distinct states bought at the cost of more total
   information. The byte sits at $m = 8$, $\Omega = 256$ — enough for a full character repertoire.

### Where this leads

Part 5 applies all of this to genomic sequences, where the alphabet is $\{A, C, T, G\}$ and the
question of why life settled on four letters becomes concrete rather than rhetorical.

---

> 🧭 **Navigation** · [← Part 3: Information Entropy of m-Blocks](../Video_03/Information_Entropy_of_m_blocks.ipynb) · **Part 4 (current)** · [Part 5: Information Entropy of a Genome →](../Video_05/Information_Entropy_of_a_Genome.ipynb)